In [ ]:
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

In [ ]:
df_data = pd.read_csv('../../data/spam-emails/combined_data.csv')
df_data.head(10)

In [ ]:
# Set up client
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/"
LLM_MODEL_NAME = "gpt-oss-20b"
client = OpenAI(
    api_key=os.getenv("OPENROUTER_LEARNING_KEY"),
    base_url=OPENROUTER_BASE_URL
)

This is a very simple pipeline. We're going to go through the first 10 rows of the dataframe, and classify it as spam or not spam, then combine the results together at the end.

In [ ]:
# Main df = df_data
# Make a copy - df_data_eval - with just 10 rows.
df_data_eval = df_data.head(10).copy()

# Set up a function to get the LLM answer simply.
def ask_llm_for_summary(email_idx: int, email_text: str, client: OpenAI, model="gpt-oss-20b", **kwargs):
    SYSTEM_PROMPT = "You are an email evaluator. Your goal is to provide a summary of the email's contents. Answer with only a summary of the email contents. You will be provided this in json: email_idx, email_text."
    
    data_dict = {
        'email_idx': email_idx,
        'email_text': email_text
    }
    
    user_prompt = json.dumps(data_dict, indent=4)
    
    llm_response = client.chat.completions.create(
        model=model,
        messages=[
            {
                'role': 'system',
                'content': SYSTEM_PROMPT
            },
            {
                'role': 'user',
                'content': user_prompt
            }
        ]
    )
    
    return llm_response

In [ ]:
# Let's try it on some text.

sample_text = df_data_eval.loc[0, ['text']].values[0]
sample_idx = df_data_eval.reset_index().loc[0, ['index']]

print(f"Sample text: {sample_text} at index {sample_idx}")

print(ask_llm_for_summary(email_idx=0, email_text=sample_text, client=client))